# CellNeighborEX v2 Tutorial — 03. Identifying Spatial Region-Specific CCI Genes in Mouse Liver Cancer (HCC)

**Key concept:** Tumor microenvironments exhibit spatially organized programs, serving as a suitable biological system for investigating spatial region-specific CCI genes and interacting cell-type pairs.

This notebook starts from bundled **precomputed `ccisignal` outputs**. We therefore skip the deconvolution step and focus on identifying spatial region-specific CCI genes and their interaction cell type pairs using `ccigenes` and `ccipairs` modules.

## Core Concept: Interpreting Spatial CCI Signals in the Tumor Microenvironment

**The tumor microenvironment (TME) exhibits spatially organized cellular composition, making it an exemplary case study for examining how spatial region-specific CCI genes and their interacting cell type pairs can be interpreted in a spatial context.**

### Using mouse liver cancer spatial transcriptomics data as an example, this tutorial demonstrates how to:

- Examine spatial cell-type distributions across tumor-associated regions
- Identify spatial region-specific CCI genes
- Infer interacting cell type pairs for each CCI gene

## ⚙️ User Setup

Before running this notebook, set `RUN_CODE_DIR` in the cell below to the path of your local `CNEXv2/run_code` directory.

**Example:**
- Linux / Mac: `Path("/home/username/CNEXv2/run_code")`
- Windows: `Path("C:/Users/username/CNEXv2/run_code")`


In [ ]:
from pathlib import Path

# ============================================================
#  USER CONFIGURATION — Edit before running
# ============================================================
RUN_CODE_DIR = Path("/path/to/CNEXv2/run_code").resolve()  # ← Change to your path
# ============================================================


In [1]:
import os
import sys

# --- Environment Configuration ---
# Set environment variables to manage threading for linear algebra libraries
# This helps prevent CPU over-subscription and ensures consistent performance
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

# --- Standard Library Imports ---
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

# --- Path Management ---
BASE_DIR = RUN_CODE_DIR.parent  # Points to the main project root (CNEXv2)

# Add the base directory to sys.path to allow importing local modules
sys.path.insert(0, str(BASE_DIR))

# --- Local Module Imports (CellNeighborEX Package) ---

# Module for cell-cell interaction (CCI) signal processing and spatial clustering
from CellNeighborEX.ccisignal import compute_proportion, cluster_spots_by_proportion

# Module for identifying and statistically validating CCI-related genes
from CellNeighborEX.ccigenes import (
    clean_column_names,
    obtain_clean_celltype_names,
    permutation_test_all_clusters,
    chi_square_goodness_of_fit,
    compute_combined_p_value,
    simplify_gene_names,
)

# Module for modeling interaction terms and validating against databases
from CellNeighborEX.ccipairs import (
    regress_residual_on_interaction_with_regularization,
    extract_interaction_terms,
    compare_database,
)

/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settin

In [2]:
# --- Data Directory Configuration ---
# Set the path for liver cancer tutorial example data
DATA_DIR = BASE_DIR / "Tutorial-ExampleData" / "3_liver_cancer"
# Single-cell RNA-seq reference file path
SC_REF_FILE = DATA_DIR / "sc_ccisignal.h5ad"
# Spatial Visium data file path
VISIUM_FILE = DATA_DIR / "sp_ccisignal.h5ad"

# --- Metadata and Analysis Settings ---
SPECIES = "mouse"
# Key for the cluster information in the AnnData object (leiden clustering on proportions)
CLUSTER_INFO = "proportion_leiden"

# --- Tutorial Execution Parameters (Speed Knobs) ---
# Number of permutations for statistical significance testing
N_PERMUTATIONS = 250
# Log Fold Change threshold for differential expression analysis
LOG_FC = 0.5
# P-value threshold for interaction terms
PVAL_TERM = 0.05

# --- Output and Figure Directory Setup ---
# Create the main output directory for liver cancer results
OUTPUT_DIR = RUN_CODE_DIR / "tutorial_output" / "03_liver_cancer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create a subdirectory specifically for saving generated plots
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# --- Scanpy Plotting Settings ---
# Set the global figure directory for Scanpy's plotting functions
sc.settings.figdir = str((OUTPUT_DIR / "figures").resolve())
# Set global resolution (dots per inch) for figures
sc.settings.set_figure_params(dpi=120)

# --- Path Verification Logging ---
# Print paths to ensure all directories and files are correctly mapped
print(f"SC_REF_FILE : {SC_REF_FILE}")
print(f"VISIUM_FILE : {VISIUM_FILE}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

SC_REF_FILE : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/Tutorial-ExampleData/3_liver_cancer/sc_ccisignal.h5ad
VISIUM_FILE : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/Tutorial-ExampleData/3_liver_cancer/sp_ccisignal.h5ad
OUTPUT_DIR  : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/run_code/tutorial_output/03_liver_cancer


## 1) Load data and check metadata information

The following datasets are loaded:
- `sc_ccisignal.h5ad`: reference signatures
- `sp_ccisignal.h5ad`: spatial deconvolution map

In [3]:
# --- Load Data ---
# Load the single-cell RNA-seq reference data (H5AD format)
adata_ref_sig = sc.read_h5ad(SC_REF_FILE)
# Load the spatial Visium data (H5AD format)
adata_vis = sc.read_h5ad(VISIUM_FILE)

# --- Summary Information ---
# Print the structure and metadata summary for both datasets
print("Reference:", adata_ref_sig)
print("Spatial:  ", adata_vis)

# --- Validation ---
# Ensure that posterior summaries (Cell2location results) are present in the .uns['mod'] slot
# These summaries are critical for downstream CellNeighborEX analysis
assert "mod" in adata_ref_sig.uns and "mod" in adata_vis.uns

# --- Metadata Standardization ---
# Check if the 'sample' column exists; if not, extract the sample name 
# from the spatial metadata dictionary to ensure consistent indexing
if "sample" not in adata_vis.obs.columns and "spatial" in adata_vis.uns:
    adata_vis.obs["sample"] = list(adata_vis.uns["spatial"].keys())[0]

# --- Preview ---
# Display the first few rows of the observation (spot) metadata for verification
display(adata_vis.obs.head())

Reference: AnnData object with n_obs × n_vars = 11683 × 14829
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'pANN_0.25_0.09_1298', 'DF.classifications_0.25_0.09_1298', 'RNA_snn_res.0.145', 'seurat_clusters', 'celltype', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'n_cells', 'nonz_mean'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'mod', 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'means_per_cluster_mu_fg', 'q05_per_cluster_mu_fg', 'q95_per_cluster_mu_fg', 'stds_per_cluster_mu_fg'
    obsp: 'distances'
Spatial:   AnnData object with n_obs × n_vars = 4706 × 13547
    obs: 'beads_per_spot', 'sample', '_indices', '_scvi_batch', '_scvi_labels', 'B', 'Hepatic Stellate Cell', 'Hepatocyte I', 'Hepatocyte II', 'Interferon', 'Kupffer/Monocyte', 'LSEC', 'Monocyte', 'Ribosomal+', 'T', 'Tumor I', 'Tumor II', 'Tumor III', 'VSMC', 'leiden', 'region_cluster', 'proportion_

,beads_per_spot,sample,_indices,_scvi_batch,_scvi_labels,B,Hepatic Stellate Cell,Hepatocyte I,Hepatocyte II,Interferon,...,leiden,region_cluster,proportion_leiden,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,mult_factor,umap_x,umap_y
50,1,artificial_Visium,0,0,0,0.000359,0.000238,0.018007,0.014692,0.000373,...,2,2,0,247,283.0,4.0,1.413428,1.678923,3762.0,680.0
188,1,artificial_Visium,1,0,0,0.000583,0.000130,0.322513,0.001768,0.000170,...,1,1,3,212,258.0,10.0,3.875969,1.631784,2442.0,800.0
191,1,artificial_Visium,2,0,0,0.001971,0.000055,0.518430,0.239212,0.000046,...,1,1,4,165,225.0,4.0,1.777778,1.424339,2622.0,800.0
193,1,artificial_Visium,3,0,0,0.000716,0.003440,0.686185,0.114666,0.000167,...,1,1,2,221,310.0,4.0,1.290323,1.779225,2742.0,800.0
194,5,artificial_Visium,4,0,0,0.005023,0.026477,4.412124,0.976698,0.001068,...,3,3,2,920,1549.0,41.0,2.646869,1.206315,2802.0,800.0


## 2) (Optional) Recompute proportions and recluster 

The spatial region clustering results (`proportion_leiden`) are already precomputed and stored in the `adata_vis.obs` of the `sp_ccisignal.h5ad` file. Therefore, you do not need to run this step. However, when starting a CellNeighborEX analysis from scratch without precomputed outputs, you should perform this step to define spatial regions (niches). This process clusters spots based on their cell-type deconvolution results (proportions) to identify distinct biological domains within the tissue.

In [4]:
# --- Spatial Proportions and Clustering ---

# Calculate the relative abundance (proportions) of each cell type 
# per spot based on the deconvolution results
adata_vis = compute_proportion(adata_vis)

# Perform neighborhood-based clustering on the cell type proportions 
# to identify distinct spatial regions (niches) within the tissue
adata_vis = cluster_spots_by_proportion(adata_vis, n_neighbors=10, resolution=0.3)

# Define the observation column that stores the resulting cluster assignments
CLUSTER_INFO = "proportion_leiden"

# --- Cluster Distribution Summary ---
# Print and display the number of spots assigned to each identified spatial cluster
print("Cluster sizes:")
display(adata_vis.obs[CLUSTER_INFO].value_counts())

Cluster sizes:


proportion_leiden
0     757
1     595
2     575
3     415
4     381
5     353
6     251
7     246
8     245
9     233
10    211
11    175
12    143
13     90
14     36
Name: count, dtype: int64

## 3) Build observed/expected expression matrices

Upon completing the `ccisignal` module, you typically obtain two primary output files: `sc_ccisignal.h5ad` and `sp_ccisignal.h5ad`. For this tutorial, these two files are provided as precomputed outputs to allow you to proceed directly with the analysis. Using these datasets, we calculate the expected expression values to compare them against the observed expression data.

In [5]:
# --- Setup and Identifier Simplification ---
cluster_info = CLUSTER_INFO

# Normalize gene names to a standard format (e.g., handling case sensitivity or prefix removal)
simplify_gene_names(adata_ref_sig, SPECIES)
simplify_gene_names(adata_vis, SPECIES)

# --- Reference Signature Extraction (Gene × Cell Type) ---
# Retrieve the list of cell type factors from the reference model
factor_names = list(adata_ref_sig.uns["mod"]["factor_names"])
# Extract the mean expression per cluster; handle different data structures (DataFrame vs. Array)
inf_raw = adata_ref_sig.varm.get("means_per_cluster_mu_fg", None)

if inf_raw is None:
    inf_aver2 = adata_ref_sig.obs[factor_names].copy()
elif isinstance(inf_raw, pd.DataFrame):
    inf_aver2 = inf_raw.copy()
else:
    inf_aver2 = pd.DataFrame(inf_raw, index=adata_ref_sig.var_names, columns=factor_names)

# Clean column names by stripping common prefixes used in modeling outputs
prefix = "means_per_cluster_mu_fg_"
if all(isinstance(c, str) and c.startswith(prefix) for c in inf_aver2.columns):
    inf_aver2.columns = [c[len(prefix):] for c in inf_aver2.columns]

# --- Feature Alignment ---
# Intersect and filter genes present in both the spatial data and the reference signature
genes = np.intersect1d(adata_vis.var_names, inf_aver2.index)
adata_vis = adata_vis[:, genes].copy()
inf_aver2 = inf_aver2.loc[genes]

# Align cell types between spatial deconvolution and reference signature
vis_factors = list(adata_vis.uns["mod"]["factor_names"])
common = [ct for ct in vis_factors if ct in inf_aver2.columns]
if not common:
    raise ValueError(f"No overlapping celltypes. vis={vis_factors}, ref={list(inf_aver2.columns)}")

# Ensure cell abundance estimates (from deconvolution) are accessible in the obs table
if not all(ct in adata_vis.obs.columns for ct in common):
    adata_vis.obs[vis_factors] = pd.DataFrame(
        adata_vis.obsm["q05_cell_abundance_w_sf"], index=adata_vis.obs_names, columns=vis_factors
    )

# --- Calculating Expected Spatial Expression ---
# Use posterior sample means to reconstruct the expected expression based on cell composition
post = adata_vis.uns["mod"]["post_sample_means"]
# dot product: Cell Abundance (Spot x CT) @ Reference Signatures (CT x Gene)
total_df = adata_vis.obs[common] @ inf_aver2[common].T

# Apply technical scaling factors: gene-specific additive effects (s_g) and spot detection efficiency (y_s)
final_df = (total_df.mul(post["m_g"], axis=1) + np.asarray(post["s_g_gene_add"])[0]).mul(post["detection_y_s"], axis=0)

# --- Preparing Dataframes for Comparison ---
meta_data = adata_vis.obs.copy()
# Combine expression values with spot metadata for both Observed and Expected datasets
observed_expression = pd.concat([adata_vis.to_df(), meta_data], axis=1)
expected_expression = pd.concat([final_df, meta_data], axis=1)

# Assign labels and adjust indices to prevent overlap during concatenation
observed_expression["condition"] = "observed"
expected_expression["condition"] = "expected"
observed_expression.index = [f"{i}_obs" for i in observed_expression.index]
expected_expression.index = [f"{i}_exp" for i in expected_expression.index]

# Create a combined AnnData object for statistical testing (Observed vs. Expected)
combined_df = pd.concat([observed_expression, expected_expression])
drop_cols = list(meta_data.columns) + ["condition"]
adata_tests = sc.AnnData(X=combined_df.drop(columns=drop_cols))
adata_tests.obs["condition"] = combined_df["condition"].values
adata_tests.obs[cluster_info] = combined_df[cluster_info].astype("category")

# --- Final Cleanup ---
# Sanitize all column and cell type names for compatibility with downstream analysis tools
observed_expression = clean_column_names(observed_expression)
expected_expression = clean_column_names(expected_expression)
meta_data = clean_column_names(meta_data)
inf_aver2 = clean_column_names(inf_aver2)
cell_types = obtain_clean_celltype_names(adata_vis)

print("OK:", observed_expression.shape, expected_expression.shape, "celltypes used:", len(common))

OK: (4706, 13577) (4706, 13577) celltypes used: 14


## 4) Detect spatial region-specific CCI genes (`ccigenes` module)

To identify genes driven by cell-cell interactions (CCI), we employ a robust statistical framework that integrates two distinct approaches: a `permutation test` to evaluate expression differences across spatial regions and a `chi-square test` to assess variance within each region. The results from both tests are combined using the `Cauchy combination test` to determine overall statistical significance. Final CCI gene candidates are then filtered based on adjusted `p-values` and `log-fold change` (LogFC) thresholds.

Note: This analytical pipeline is consistent across various datasets; only the `SPECIES` and `CLUSTER_INFO` parameters require adjustment to match the specific sample metadata.

In [6]:
# --- Statistical Testing for CCI Gene Identification ---

# Run permutation test across all spatial clusters to assess 
# expression differences between observed and expected data
perm_results = permutation_test_all_clusters(
    adata_tests,
    cluster_info=cluster_info,
    observed_expression=observed_expression,
    expected_expression=expected_expression,
    n_permutations=N_PERMUTATIONS,
    use_zeros=True,
    random_seed=42,
    path=str(OUTPUT_DIR),
)

# Perform Chi-square goodness-of-fit test within each cluster 
# to evaluate the variance between observed and expected distributions
chi_results = chi_square_goodness_of_fit(
    adata_tests,
    cluster_info=cluster_info,
    groupby="condition",
    reference="observed",
    target="expected",
    use_zeros=True,
)

# --- Statistical Integration ---

# Merge permutation and chi-square results for each gene-cluster pair
stats_df = pd.merge(perm_results, chi_results, on=["gene", "cluster"], how="inner")

# Combine p-values from both tests using the Cauchy combination test
stats_df["combined_p_value_adj"] = stats_df.apply(compute_combined_p_value, axis=1)

# Save the comprehensive statistics for all tested genes
stats_df.to_csv(OUTPUT_DIR / "allgenes_results.csv", index=False)

# --- Filtering for Significant CCI Genes ---

# Filter genes based on adjusted p-value, higher observed vs. expected expression, 
# and a minimum log-fold change (LogFC) threshold
gene_filter = (
    (stats_df["combined_p_value_adj"] < 0.01)
    & (stats_df["mean_ref"] > stats_df["mean_tgt"])
    & (stats_df["logfc"] > LOG_FC)
)
final_results = stats_df.loc[gene_filter].copy()

# Save the final list of identified CCI genes (niche-specific genes)
final_results.to_csv(OUTPUT_DIR / "ccigenes_results.csv", index=False)

# --- Results Summary and Display ---

# Display the total count and the top 20 CCI gene candidates
print("Niche gene hits:", final_results.shape[0])
display(final_results[["gene", "cluster", "combined_p_value_adj", "logfc"]].head(20))

# Group by spatial cluster and identify the top 10 most significant genes per region
print("Top 10 niche genes per cluster (by adjusted p):")
top_by_cluster = (
    final_results.sort_values(["cluster", "combined_p_value_adj"]) 
    .groupby("cluster")
    .head(10)
)
display(top_by_cluster[["cluster", "gene", "combined_p_value_adj", "logfc"]])

Permutation Test Progress for Cluster 14: 100%|██████████| 13547/13547 [00:59<00:00, 226.26it/s]


Performing Peason's Chi-Square Test for cluster 0...


Chi-Square Test Progress: 100%|██████████| 13547/13547 [00:13<00:00, 1000.20it/s]


Performing Peason's Chi-Square Test for cluster 1...


Chi-Square Test Progress: 100%|██████████| 13547/13547 [00:12<00:00, 1059.13it/s]


...

Chi-Square Test Progress: 100%|██████████| 13547/13547 [00:09<00:00, 1376.25it/s]


Niche gene hits: 1742


,gene,cluster,combined_p_value_adj,logfc
564,Actb,0,2.000000e-299,0.619224
1714,C3,0,2.000000e-299,0.532355
2661,Cp,0,2.000000e-300,0.580960
2779,Cst3,0,2.000000e-299,0.541184
2817,Ctsb,0,2.000000e-299,0.591537
2819,Ctsd,0,2.000000e-299,0.621400
5221,H2_Aa,0,2.000000e-299,1.408171
5222,H2_Ab1,0,2.000000e-299,1.105903
5227,H2_Eb1,0,2.000000e-299,0.687807
5229,H2_K1,0,2.000000e-299,1.532469


Top 10 niche genes per cluster (by adjusted p):


,cluster,gene,combined_p_value_adj,logfc
2661,0,Cp,2.000000e-300,0.580960
5288,0,Hbb_bs,2.000000e-300,1.346579
7232,0,Mup20,2.000000e-300,1.395840
5479,0,Hp,2.222222e-300,4.206508
5494,0,Hpx,2.222222e-300,2.462523
...,...,...,...,...
117997,9,H3f3b,2.000000e-299,1.737508
118670,9,Kazald1,2.000000e-299,0.735342
118877,9,Krt79,2.000000e-299,0.798386
118970,9,Lgals1,2.000000e-299,1.116304


## 5) Identify interacting cell-type pairs (`ccipairs` module)

Following the identification of CCI genes, we determine the specific interacting cell-type pairs through a `regression-based framework` (a two-step modeling strategy). In this step, we regress the residual expression signals against cell-type interaction terms to quantify pairwise effects. Finally, these significant interaction terms are biologically annotated by mapping them to curated ligand-receptor databases(`Omnipath`, `CellChat`, `CellTalk`), bridging the gap between observed spatial signals and established molecular signaling pathways.

In [7]:
# --- Identify Interacting Cell-Type Pairs (ccipairs) ---

# Check if any significant CCI (niche) genes were found in the previous step
if final_results.empty:
    print("No niche genes detected; skipping ccipairs.")
else:
    # Normalize cell-type deconvolution values so that the sum per spot equals 1
    norm_deconv = meta_data.loc[:, cell_types].div(meta_data.loc[:, cell_types].sum(axis=1), axis=0)
    norm_deconv[cluster_info] = meta_data[cluster_info]
    
    # Calculate the mean cell-type composition for each spatial cluster (niche)
    cluster_summary = norm_deconv.groupby(cluster_info).mean()
    cluster_summary.loc["mean"] = cluster_summary.mean()

    # Data cleanup: Remove duplicate genes from the reference signature and results
    inf_aver2 = inf_aver2[~inf_aver2.index.duplicated(keep="first")]
    final_results = final_results.drop_duplicates(subset=["gene"])

    # --- Two-Step Regression Modeling ---
    # Regress the residual signal (Observed - Expected) onto cell-type interaction terms.
    # We use Elastic Net regularization (alpha=0.5) to handle multi-collinearity.
    models_per_gene, gene_analysis = regress_residual_on_interaction_with_regularization(
        observed_expression=observed_expression,
        expected_expression=expected_expression,
        celltypes=cell_types,
        cell_sig=inf_aver2,
        niche_gene_results=final_results,
        cluster_summary=cluster_summary,
        cluster_info=cluster_info,
        self_interaction=False, # Exclude self-interactions (e.g., CellTypeA-CellTypeA)
        use_zeros=False,
        alpha=0.5,
    )

    # --- Extract Significant Interactions ---
    # Filter for interaction terms that meet the statistical p-value threshold
    interaction_terms = extract_interaction_terms(
        models_per_gene, gene_analysis, p_value_threshold=PVAL_TERM
    )

    if interaction_terms.empty:
        print("No significant interaction terms.")
    else:
        # For each gene in each cluster, keep the top 5 most significant cell-type pairs
        interaction_terms = (
            interaction_terms.groupby(["cluster", "gene"]) 
            .apply(lambda df: df.sort_values("p_value").head(5))
            .reset_index(drop=True)
        )
        
        # --- Biological Validation ---
        # Map the identified cell-type pairs and genes to curated Ligand-Receptor databases
        interaction_terms = compare_database(interaction_terms, species=SPECIES)
        
        # Save final results containing gene, interacting pairs, and database evidence
        interaction_terms.to_csv(OUTPUT_DIR / "ccipairs_results.csv", index=False)

        # Display the most significant validated interaction results
        print("Top interaction terms (head):")
        display(interaction_terms.head(25))

Processing Genes:  67%|██████▋   | 372/553 [08:41<04:14,  1.41s/gene]

Skipping single regression for Bhmt, variable Interferon_T, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable Hepatocyte_II_Interferon, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable LSEC_Tumor_III, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable T_Tumor_III, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable Hepatocyte_II_Tumor_III, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, vari

Processing Genes:  67%|██████▋   | 373/553 [08:42<03:57,  1.32s/gene]

Skipping single regression for Bhmt, variable Hepatocyte_I_Monocyte, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable Interferon_Kupffer_Monocyte, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable Hepatic_Stellate_Cell_Monocyte, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable Hepatocyte_I_LSEC, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single regression for Bhmt, variable Hepatocyte_I_T, cluster 6: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.
Skipping single 

Processing Genes:  68%|██████▊   | 374/553 [08:43<03:48,  1.28s/gene]

...

Processing Genes: 100%|██████████| 553/553 [10:33<00:00,  1.15s/gene]


Skipping Bhmt in cluster 6 due to missing models.
Skipping Cd302 in cluster 6 due to missing models.
Skipping Chchd10 in cluster 6 due to missing models.
Skipping Dbi in cluster 6 due to missing models.
Skipping Gnmt in cluster 6 due to missing models.
Skipping Pemt in cluster 6 due to missing models.
Skipping Prdx6 in cluster 6 due to missing models.
Skipping Serpina1a in cluster 6 due to missing models.
Skipping Tdo2 in cluster 6 due to missing models.


Processing Queries: 100%|██████████| 967/967 [00:11<00:00, 86.57it/s] 

Top interaction terms (head):


,gene,cluster,term,index_celltype,neighboring_celltype,coef,std_err,p_value,source_omnipath,pair_omnipath,source_cellchat,pair_cellchat,source_celltalk,pair_celltalk
0,Actb,0,Monocyte_VSMC,Monocyte,VSMC,0.225725,0.035245,1.508726e-10,omnipath,single KDM1A-ACTB; single NFIC-ACTB; single CK...,unknown,unknown,unknown,unknown
1,Actb,0,B_Monocyte,B,Monocyte,0.788765,0.130633,1.560129e-09,omnipath,single KDM1A-ACTB; single NFIC-ACTB; single CK...,unknown,unknown,unknown,unknown
2,Actb,0,Monocyte_T,Monocyte,T,0.231596,0.038660,2.090546e-09,omnipath,single KDM1A-ACTB; single NFIC-ACTB; single CK...,unknown,unknown,unknown,unknown
3,Actb,0,B_T,B,T,2.443371,0.442041,3.248799e-08,omnipath,single KDM1A-ACTB; single NFIC-ACTB; single CK...,unknown,unknown,unknown,unknown
4,Actb,0,Interferon_Monocyte,Monocyte,Interferon,0.992943,0.183639,6.407115e-08,omnipath,single KDM1A-ACTB; single NFIC-ACTB; single CK...,unknown,unknown,unknown,unknown
5,C3,0,Hepatocyte_II_VSMC,Hepatocyte_II,VSMC,0.245932,0.032405,3.217586e-14,omnipath,single C3-C3AR1; single CFI-C3; single CTSG-C3...,cellchat,single C3-C3AR1; single C3-CR2; single C3-ITGA...,celltalk,single C3-CD46; single C3-CD55; single C3-LRP1...
6,C3,0,Hepatocyte_I_VSMC,Hepatocyte_I,VSMC,0.119448,0.020110,2.852437e-09,omnipath,single C3-C3AR1; single CFI-C3; single CTSG-C3...,cellchat,single C3-C3AR1; single C3-CR2; single C3-ITGA...,celltalk,single C3-CD46; single C3-CD55; single C3-LRP1...
7,C3,0,Hepatocyte_I_Hepatocyte_II,Hepatocyte_II,Hepatocyte_I,0.020163,0.003502,8.555537e-09,omnipath,single C3-C3AR1; single CFI-C3; single CTSG-C3...,cellchat,single C3-C3AR1; single C3-CR2; single C3-ITGA...,celltalk,single C3-CD46; single C3-CD55; single C3-LRP1...
8,C3,0,Hepatocyte_II_Monocyte,Hepatocyte_II,Monocyte,0.071663,0.013132,4.843077e-08,omnipath,single C3-C3AR1; single CFI-C3; single CTSG-C3...,cellchat,single C3-C3AR1; single C3-CR2; single C3-ITGA...,celltalk,single C3-CD46; single C3-CD55; single C3-LRP1...
9,C3,0,Hepatocyte_II_Ribosomal_,Hepatocyte_II,Ribosomal_,0.329240,0.074056,8.755489e-06,omnipath,single C3-C3AR1; single CFI-C3; single CTSG-C3...,cellchat,single C3-C3AR1; single C3-CR2; single C3-ITGA...,celltalk,single C3-CD46; single C3-CD55; single C3-LRP1...
